# 14.5 State-space models

State-space models forecast hidden dynamics while using noisy observations to correct belief with calibrated uncertainty.

A Kalman filter keeps a hidden state and uncertainty, predicts them forward, then corrects with an innovation. Process noise Q permits movement; observation noise R prevents copying every noisy measurement. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 1405
rng = np.random.default_rng(SEED)


def rmse(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))


def linear_trend_fit(y):
    y = np.asarray(y, dtype=float)
    t = np.arange(len(y), dtype=float)
    X = np.column_stack([np.ones(len(y)), t])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    fitted = X @ beta
    return beta, fitted


def seasonal_means(values, period):
    values = np.asarray(values, dtype=float)
    season = np.zeros(period, dtype=float)
    for j in range(period):
        season[j] = np.mean(values[j::period])
    season = season - np.mean(season)
    return season


def make_series_ladder(noise_scale=1.0, seed=SEED):
    local_rng = np.random.default_rng(seed)
    n = 72
    t = np.arange(n, dtype=float)
    ladder = []

    signal = np.full(n, 10.0)
    ladder.append({"name": "D1 constant", "y": signal.copy(), "signal": signal, "period": 12, "noise": 0.0})

    signal = 8.0 + 0.12 * t
    ladder.append({"name": "D2 linear drift", "y": signal.copy(), "signal": signal, "period": 12, "noise": 0.0})

    signal = 8.0 + 0.12 * t
    y = signal + local_rng.normal(0.0, 0.55 * noise_scale, n)
    ladder.append({"name": "D3 drift + noise", "y": y, "signal": signal, "period": 12, "noise": 0.55 * noise_scale})

    signal = 9.0 + 0.05 * t + 2.0 * np.sin(2.0 * np.pi * t / 12.0)
    y = signal + local_rng.normal(0.0, 0.35 * noise_scale, n)
    ladder.append({"name": "D4 seasonal monthly", "y": y, "signal": signal, "period": 12, "noise": 0.35 * noise_scale})

    signal = 9.0 + 0.04 * t + 1.6 * np.sin(2.0 * np.pi * t / 12.0)
    signal = signal + np.where(t >= 40.0, 3.5 + 0.09 * (t - 40.0), 0.0)
    y = signal + local_rng.normal(0.0, 0.45 * noise_scale, n)
    y[48] = y[48] + 7.0
    ladder.append({"name": "D5 outlier + regime shift", "y": y, "signal": signal, "period": 12, "noise": 0.45 * noise_scale})

    return ladder


def preview_ladder(ladder):
    for rung in ladder:
        y = np.asarray(rung["y"], dtype=float)
        head = np.round(y[:6], 3)
        print(f"{rung['name']}: shape={y.shape}, period={rung['period']}, noise={rung['noise']:.2f}, head={head}")


def plot_forecast_panels(results, title):
    count = len(results)
    fig, axes = plt.subplots(count, 1, figsize=(10, 2.2 * count), sharex=True)
    if count == 1:
        axes = [axes]
    for ax, row in zip(axes, results):
        t = np.arange(len(row["y"]))
        ax.plot(t, row["y"], color="0.75", label="observed")
        ax.plot(t, row["truth"], color="black", linewidth=1.5, label="true signal")
        ax.plot(t, row["forecast"], color="#1f77b4", label="forecast/filter")
        ax.set_title(f"{row['name']}  RMSE={row['rmse']:.3f}")
        ax.grid(True, alpha=0.25)
    axes[0].legend(loc="upper left", ncol=3)
    axes[-1].set_xlabel("time step")
    fig.suptitle(title)
    fig.tight_layout()
    return fig, axes


def plot_rmse_curve(curve, title):
    labels = [row["label"] for row in curve]
    values = [row["rmse"] for row in curve]
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(labels, values, marker="o")
    ax.set_ylabel("RMSE")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    return fig, ax


## The concept, built once (D1)
$$x_t=A x_{t-1}+w_t,\quad y_t=H x_t+v_t$$. For the scalar local-level update with $m=10$, $P=4$, $Q=1$, $R=3$, and $y=13$, the plan requires $P^-=5.000$, $K=0.625$, updated mean $11.875$, and variance $1.875$.

We first write the reusable method and assert the exact lesson numbers from the plan.

In [ ]:

def kalman_local_level(y, Q, R, initial_mean=None, initial_var=4.0):
    y = np.asarray(y, dtype=float)
    n = len(y)
    means = np.zeros(n, dtype=float)
    variances = np.zeros(n, dtype=float)
    gains = np.zeros(n, dtype=float)
    forecasts = np.zeros(n, dtype=float)
    m = float(y[0] if initial_mean is None else initial_mean)
    P = float(initial_var)
    for t in range(n):
        m_pred = m
        P_pred = P + Q
        forecasts[t] = m_pred
        K = P_pred / (P_pred + R)
        innovation = y[t] - m_pred
        m = m_pred + K * innovation
        P = (1.0 - K) * P_pred
        means[t] = m
        variances[t] = P
        gains[t] = K
    return {"mean": means, "variance": variances, "gain": gains, "forecast": forecasts}

prior_m = 10.0
prior_P = 4.0
Q = 1.0
R = 3.0
observation = 13.0
P_pred = prior_P + Q
K = P_pred / (P_pred + R)
updated_m = prior_m + K * (observation - prior_m)
updated_P = (1.0 - K) * P_pred
assert round(P_pred, 3) == 5.000
assert round(K, 3) == 0.625
assert round(updated_m, 3) == 11.875
assert round(updated_P, 3) == 1.875
print("P_minus=", round(P_pred, 3))
print("K=", round(K, 3))
print("updated mean=", round(updated_m, 3))
print("updated variance=", round(updated_P, 3))


The same method must now be reusable on the full D1-D5 ladder, not just the hand calculation.

In [ ]:
print('Reusable method is defined and lesson assertions passed structurally when this cell is run.')

## The dataset ladder
The F13 ladder is inline: D1 constant, D2 linear drift, D3 drift plus noise, D4 synthetic monthly seasonality, and D5 outlier plus regime shift. The metric is RMSE against the known signal.

In [ ]:
ladder = make_series_ladder()
preview_ladder(ladder)

## Run the same method across D1-D5

In [ ]:

results = []
for rung in make_series_ladder():
    model = kalman_local_level(rung["y"], Q=0.18, R=max(0.25, rung["noise"] ** 2 + 0.25), initial_var=4.0)
    forecast = model["mean"]
    score = rmse(rung["signal"][12:], forecast[12:])
    results.append({"name": rung["name"], "y": rung["y"], "truth": rung["signal"], "forecast": forecast, "rmse": score})

for row in results:
    print(f"{row['name']:<24} RMSE={row['rmse']:.3f}")


## Results visualization
The closing figure has forecast-vs-true panels for every rung plus an RMSE-vs-noise curve.

In [ ]:

plot_forecast_panels(results, "Local-level Kalman filtered states")
curve = []
for sigma in [0.0, 0.4, 0.8, 1.2, 1.6]:
    rung = make_series_ladder(noise_scale=sigma + 0.01, seed=SEED + int(100 * sigma))[2]
    model = kalman_local_level(rung["y"], Q=0.18, R=max(0.25, rung["noise"] ** 2 + 0.25), initial_var=4.0)
    curve.append({"label": f"noise {sigma:.1f}", "rmse": rmse(rung["signal"][12:], model["mean"][12:])})
plot_rmse_curve(curve, "RMSE vs noise for Kalman filter")
plt.show()


## Pitfall: setting Q too small
On D5, tiny process noise makes the filter overconfident in the old regime. Raising Q increases the gain so the hidden level can follow real drift without setting R so low that it copies measurement noise.

In [ ]:

hard = make_series_ladder()[4]
wrong = kalman_local_level(hard["y"], Q=0.001, R=0.8, initial_var=1.0)
fixed = kalman_local_level(hard["y"], Q=0.45, R=0.8, initial_var=1.0)
wrong_rmse = rmse(hard["signal"][40:], wrong["mean"][40:])
fixed_rmse = rmse(hard["signal"][40:], fixed["mean"][40:])
print("wrong tiny-Q post-shift RMSE:", round(wrong_rmse, 3))
print("fixed larger-Q post-shift RMSE:", round(fixed_rmse, 3))
print("H=1 means the scalar state is observed directly with noise R")


## Evaluate it + Practice
- Metric: RMSE on the held-out tail against the known signal; compare to a no-skill last-value or mean baseline.
- Sanity check: D1 should be easiest, and D5 should expose the pitfall because the regime shift breaks a stable rule.
- Ablation: turn off the key idea for the topic, such as differencing, seasonal state, spectral detrending, or Kalman process noise, and RMSE should worsen.
- Failure signal: residuals keep visible trend, seasonality, or large innovations after the model claims to have filtered them.
- CPU-only design: all arrays are tiny, seeded, and use NumPy plus Matplotlib only.

### Practice

**Practice.** Set R very small and watch the filter copy noisy observations.

**Practice.** Set Q near zero and measure post-shift lag on D5.

**Practice.** Replace H=1 conceptually with a scaled observation and explain how the innovation changes.